In [1]:
import joblib

In [2]:
model = joblib.load("fraud_detection_pipeline.pkl")

In [3]:
import pandas as pd

sample = pd.DataFrame([{
    "type": "PAYMENT",
    "amount": 5000000000,
    "oldbalanceOrg": 5000000000, # sender
    "newbalanceOrig": 0,     #sender
    "oldbalanceDest": 10,    # receiver
    "newbalanceDest": 5000000010        #receiver
}])

prediction = model.predict(sample)

In [4]:
if prediction == 1:
    print("Transaction might be FRAUD!")
else:
    print("CLEAN!")

Transaction might be FRAUD!


In [8]:
sample = pd.DataFrame([{
    "type": "PAYMENT",
    "amount": 50000000, 
    "oldbalanceOrg": 1000000000, # sender
    "newbalanceOrig": 950000000,     #sender
    "oldbalanceDest": 10000,    # receiver
    "newbalanceDest": 50010000        #receiver
}])

prediction = model.predict(sample)

In [9]:
if prediction == 1:
    print("Transaction might be FRAUD!")
else:
    print("CLEAN!")

Transaction might be FRAUD!


In [18]:
sample = pd.DataFrame([{
    "type": "PAYMENT",
    "amount": 50000, 
    "oldbalanceOrg": 100000, # sender
    "newbalanceOrig": 50000,     #sender
    "oldbalanceDest": 50000,    # receiver
    "newbalanceDest": 100000        #receiver
}])

prediction = model.predict(sample)

In [19]:
if prediction == 1:
    print("Transaction might be FRAUD!")
else:
    print("CLEAN!")

CLEAN!


In [30]:
import pandas as pd

test_cases = pd.DataFrame([
    # ---- NORMAL TRANSACTIONS (Expected: 0) ----
    {"type": "PAYMENT", "amount": 500,   "oldbalanceOrg": 5000,  "newbalanceOrig": 4500, "oldbalanceDest": 0,     "newbalanceDest": 500},
    {"type": "PAYMENT", "amount": 1200,  "oldbalanceOrg": 15000, "newbalanceOrig": 13800,"oldbalanceDest": 0,     "newbalanceDest": 1200},
    {"type": "CASH_IN", "amount": 3000,  "oldbalanceOrg": 2000,  "newbalanceOrig": 5000, "oldbalanceDest": 10000, "newbalanceDest": 13000},

    # ---- SUSPICIOUS / FRAUD-LIKE (Expected: 1) ----
    # money emptied from account
    {"type": "TRANSFER", "amount": 9500, "oldbalanceOrg": 9500, "newbalanceOrig": 0,    "oldbalanceDest": 0,     "newbalanceDest": 0},

    # cash out after transfer pattern
    {"type": "CASH_OUT", "amount": 8000, "oldbalanceOrg": 8000, "newbalanceOrig": 0,    "oldbalanceDest": 3000,  "newbalanceDest": 11000},

    # large amount, destination not updated
    {"type": "TRANSFER", "amount": 15000,"oldbalanceOrg": 16000,"newbalanceOrig": 1000, "oldbalanceDest": 0,     "newbalanceDest": 0},

    # ---- EDGE CASES ----
    {"type": "PAYMENT", "amount": 10,    "oldbalanceOrg": 100,  "newbalanceOrig": 90,   "oldbalanceDest": 0,     "newbalanceDest": 10},
    {"type": "TRANSFER","amount": 50000, "oldbalanceOrg": 50000,"newbalanceOrig": 0,    "oldbalanceDest": 0,     "newbalanceDest": 0},
])


In [32]:
pred = model.predict(test_cases)
proba = model.predict_proba(test_cases)

results = test_cases.copy()
results["Prediction"] = ["Fraud" if p == 1 else "Not Fraud" for p in pred]
results["Fraud_Probability"] = (proba[:, 1] * 100).round(2)  # in %

results


,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,Prediction,Fraud_Probability
0,PAYMENT,500,5000,4500,0,500,Not Fraud,0.00
1,PAYMENT,1200,15000,13800,0,1200,Not Fraud,0.00
2,CASH_IN,3000,2000,5000,10000,13000,Not Fraud,0.00
3,TRANSFER,9500,9500,0,0,0,Fraud,79.94
4,CASH_OUT,8000,8000,0,3000,11000,Not Fraud,41.90
5,TRANSFER,15000,16000,1000,0,0,Fraud,82.06
6,PAYMENT,10,100,90,0,10,Not Fraud,0.00
7,TRANSFER,50000,50000,0,0,0,Fraud,91.68


In [33]:
extra_cases = pd.DataFrame([

    # -------- NORMAL BEHAVIOR --------
    {"type": "PAYMENT", "amount": 250,   "oldbalanceOrg": 3000,  "newbalanceOrig": 2750, "oldbalanceDest": 0,     "newbalanceDest": 250},
    {"type": "CASH_IN", "amount": 10000, "oldbalanceOrg": 5000,  "newbalanceOrig": 15000,"oldbalanceDest": 20000, "newbalanceDest": 30000},
    {"type": "PAYMENT", "amount": 75,    "oldbalanceOrg": 500,   "newbalanceOrig": 425,  "oldbalanceDest": 0,     "newbalanceDest": 75},
    {"type": "DEBIT",   "amount": 600,   "oldbalanceOrg": 5000,  "newbalanceOrig": 4400, "oldbalanceDest": 0,     "newbalanceDest": 600},

    # -------- FRAUD-LIKE PATTERNS --------
    # full balance transfer
    {"type": "TRANSFER", "amount": 20000, "oldbalanceOrg": 20000, "newbalanceOrig": 0, "oldbalanceDest": 0, "newbalanceDest": 0},

    # destination balance unchanged
    {"type": "CASH_OUT", "amount": 12000, "oldbalanceOrg": 13000, "newbalanceOrig": 1000, "oldbalanceDest": 5000, "newbalanceDest": 5000},

    # repeated high-risk pattern
    {"type": "TRANSFER", "amount": 7000, "oldbalanceOrg": 7000, "newbalanceOrig": 0, "oldbalanceDest": 0, "newbalanceDest": 0},

    # inconsistent balances
    {"type": "CASH_OUT", "amount": 4000, "oldbalanceOrg": 4500, "newbalanceOrig": 500, "oldbalanceDest": 2000, "newbalanceDest": 2000},

    # -------- EDGE / TRICKY CASES --------
    {"type": "PAYMENT", "amount": 9999, "oldbalanceOrg": 20000, "newbalanceOrig": 10001, "oldbalanceDest": 0, "newbalanceDest": 9999},
    {"type": "TRANSFER","amount": 1,    "oldbalanceOrg": 1,     "newbalanceOrig": 0,     "oldbalanceDest": 0, "newbalanceDest": 0},
])


In [34]:
all_test_cases = pd.concat([test_cases, extra_cases], ignore_index=True)

pred = model.predict(all_test_cases)
proba = model.predict_proba(all_test_cases)

results = all_test_cases.copy()
results["Prediction"] = ["Fraud" if p == 1 else "Not Fraud" for p in pred]
results["Fraud_Probability (%)"] = (proba[:, 1] * 100).round(2)

results


,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,Prediction,Fraud_Probability (%)
0,PAYMENT,500,5000,4500,0,500,Not Fraud,0.00
1,PAYMENT,1200,15000,13800,0,1200,Not Fraud,0.00
2,CASH_IN,3000,2000,5000,10000,13000,Not Fraud,0.00
3,TRANSFER,9500,9500,0,0,0,Fraud,79.94
4,CASH_OUT,8000,8000,0,3000,11000,Not Fraud,41.90
5,TRANSFER,15000,16000,1000,0,0,Fraud,82.06
6,PAYMENT,10,100,90,0,10,Not Fraud,0.00
7,TRANSFER,50000,50000,0,0,0,Fraud,91.68
8,PAYMENT,250,3000,2750,0,250,Not Fraud,0.00
9,CASH_IN,10000,5000,15000,20000,30000,Not Fraud,0.00
